# Lesson 5: Advanced Sensing

In this lesson, we’ll learn to use sensors that transmit complex information using communication protocols. This includes:
- Understanding **communication protocols**
- Using the **I2C** protocol with an **Inertial Measurement Unit (IMU)** for navigation
- Building a **Finite State Machine (FSM)** for obstacle avoidance


## What is a Communication Protocol?

Up to now we’ve thought primarily about a robot with one onboard processor (our Arduino)  acting as the robot’s brain, managing a lot of other simpler components. Yet, many complicated robots are networks of different  processors each managing their own complicated tasks (e.g. one processor dedicated to motion  planning and decision-making, one dedicated to sensing the environment, and another dedicated  to controlling actuators).

A communication protocol is a method that enables processors to exchange data effectively. In complex robotics, where processors handle specialized tasks like motion planning, sensing, or controlling actuators, protocols ensure that all processors coordinate smoothly.

- **Digital Communication**: Binary (`0` and `1`) values are sent over connections, allowing processors to interpret and share data.

Digital data is represented with a [base-2](https://learn.sparkfun.com/tutorials/binary)  number system, where each digit- or “bit”- can have a value of either 0 or 1. It’s easy enough to  assign a value of 0 to a LOW pin voltage (0V) and a value of 1 to a HIGH pin voltage (5V, in the  case of the Arduino). By setting our pin voltage LOW we can send a 0 and by setting it HIGH we can send a 1. The thing is [integers other than 0 and 1](https://www.purplemath.com/modules/numbbase.htm), [floating point numbers](https://towardsdatascience.com/binary-representation-of-the-floating-point-numbers-77d7364723f1), and [alphanumeric characters](https://www.geeksforgeeks.org/ascii-table/) are all represented with groups of bits, so we need to send more than  one bit in a message.
 
- **Serial Communication**: Data is sent bit-by-bit to save wiring, using either a clock line (synchronous) or timing conventions (asynchronous) to sync data transmission.

## Serial Communication Fundamentals

### 1. Synchronous Serial Communication
In synchronous communication, a **clock line** (signal wire) synchronizes data timing. The data line only needs to change state when the clock pulses, simplifying data interpretation for receiving devices.  Assuming our clock is rising edge active (indicated by the arrow on the rising edge), each time we raise the [logic level](https://learn.sparkfun.com/tutorials/logic-levels) on our  clock to HIGH, the receiver knows to read a bit from the data line.


![Inter-processor communication using a clock wire to synchronize the message](5.1.png)

### 2. Asynchronous Serial Communication
In asynchronous communication, there’s no clock line. Instead, devices synchronize through specific timing structures and start and stop bits. This method is slower but simplifies hardware connections. The data line is set to HIGH in its idle state (referred to  as its marking state). The beginning of a message is indicated when the data line drops from the  marking state at HIGH to the spacing state at LOW, which marks the beginning of the start bit.  The start bit lasts 1 bit-time, after which the sender transmits all the bits in the message, each lasting 1 bit-time. Typically, these messages will be 8 bits long, though seven is also common. The first bit transferred is the [least significant bit](https://en.wikipedia.org/wiki/Bit_numbering) (LSB), while the final bit is the [most significant  bit](https://en.wikipedia.org/wiki/Bit_numbering) (MSB). After the MSB, the stop bit is sent to signal the end of the message. After the stop bit  has been sent, the data line is again free to send another message.


![Inter-processor communication without a clock wire ](5.2.png)


## Protocols Overview: I2C, SPI, and UART

In robotics, three main protocols are popular for enabling communication between various components and processors: **I2C**, **SPI**, and **UART**. Each has unique characteristics that make it suitable for specific applications.

**I2C ([Inter-Integrated Circuit](https://learn.sparkfun.com/tutorials/i2c/all))**
- **Type**: Synchronous serial communication
- **Structure**: Uses two lines (SDA for data and SCL for clock) to connect multiple devices.
- **Purpose**: Ideal for systems where several components need to communicate over a single bus with minimal wiring.
- **Usage**: Often used for sensors, displays, and modules that need limited bandwidth but multiple device connections.

**SPI ([Serial Peripheral Interface](https://learn.sparkfun.com/tutorials/serial-peripheral-interface-spi/all))**
- **Type**: Synchronous serial communication
- **Structure**: Uses four lines (MISO, MOSI, SCK, and CS) to support high-speed communication between a controller and peripherals.
- **Purpose**: Best for applications needing fast data transfer between a few devices.
- **Usage**: Common for memory devices, high-speed sensors, and displays requiring fast updates.

**UART ([Universal Asynchronous Receiver-Transmitter](https://docs.arduino.cc/learn/communication/uart/#rxtx-pin-examples))**
- **Type**: Asynchronous serial communication
- **Structure**: Requires only two data lines (TX and RX) with no clock line, simplifying connections.
- **Purpose**: Simplifies communication between two devices but is limited in speed and scalability.
- **Usage**: Widely used for simple, low-speed serial communication, such as debugging and connecting microcontrollers with a computer.


## Inter-Integrated Circuit Bus (I2C)

**I2C** is a popular synchronous serial communication protocol that allows multiple devices to communicate with a controller on the same bus with only two main lines. Known for its speed and simplicity, I2C can connect more devices than asynchronous protocols while using fewer wires than other synchronous options.

### Setting Up the I2C Bus

To connect devices over I2C:
- **SDA (Serial Data Line)**: All devices connect to this line to share data.
- **SCL (Serial Clock Line)**: Used for timing signals, shared by all devices on the bus.
  
> **Note**: All devices need a common ground for accurate communication. This is often omitted in diagrams but is crucial for functionality.

![The basic I2C circuit schematic](5.3.png)

### Structure of I2C Messages

An I2C message has several parts:
1. **Address Frame**: The controller uses this 7-bit address to select the target device.
2. **Data Frames**: Following the address, each 8-bit data frame carries the actual message content. These data frames are either sent to or requested from the device.

The controller initiates all communication and decides when to send or receive data from a device.


### I2C Bus Configuration and Pull-Up Resistors

The **SDA** and **SCL** lines in I2C idle at a HIGH logic level, achieved through **pull-up resistors**. In an Arduino circuit:
- Pull-up resistors are usually built-in, making circuit assembly easier.
  
These resistors ensure that data transmission is stable and consistent across the devices on the I2C bus.

### I2C Communication with Arduino: The Wire Library

The Arduino **Wire library** simplifies communication over I2C. This library provides functions to send and receive data frames from devices on the I2C bus. To communicate with most I2C devices, custom libraries are often available that build on the Wire library, adding sensor-specific functions and making setup easier.

> For additional examples, see [Wire Library Documentation](https://www.arduino.cc/en/Reference/Wire).

### SPI (Serial Peripheral Interface)

SPI is a high-speed, synchronous protocol widely used in electronics, especially in applications requiring fast data transfer. Although SPI requires more hardware than I2C, it provides even faster communication and higher speeds, making it ideal for use with devices like memory modules, high-speed sensors, and displays. 

**Key Characteristics:**
- **Communication Lines**: SPI uses four main lines:
  - **SCK** (Clock) – Controls the timing of data transmission.
  - **COPI** (Controller Out, Peripheral In) – Data sent from the controller to the peripheral.
  - **CIPO** (Controller In, Peripheral Out) – Data sent from the peripheral to the controller.
  - **CS** (Chip Select) – Each device has its CS pin, and only the device with a pulled-low CS pin is active and communicates with the controller.

- **Device Selection**: Each SPI peripheral device communicates only when its **CS** pin is pulled LOW. If CS is HIGH, the device ignores data from the controller. This setup makes selecting devices straightforward but means that the controller must have a CS line for each peripheral it communicates with.

![The basic SPI circuit schematic](5.4.png)

---

**Communication Terminology Update**  
SPI terminology has shifted towards more inclusive language, with **controller** and **peripheral** replacing traditional master/slave terms. You may encounter both sets of terms in documentation:

| **Old Terms**                | **New Terms**                       |
|------------------------------|--------------------------------------|
| Master in, slave out (MISO)  | Controller in, peripheral out (CIPO) |
| Master out, slave in (MOSI)  | Controller out, peripheral in (COPI) |
| Slave select (SS)            | Chip Select (CS)                    |

**Message Structure**  
SPI’s message structure is generally simpler than I2C, allowing for faster data transmission. However, this simplicity is achieved by relying more on hardware, such as dedicated lines for COPI and CIPO, rather than shared data lines.

**Programming SPI in Arduino**  
Arduino’s SPI library offers an easy way to handle SPI communication. It provides functions that streamline the process of sending and receiving data over SPI, making SPI accessible for fast communication projects with Arduino.


### UART (Universal Asynchronous Receiver-Transmitter)

UART is an asynchronous protocol designed for simple, point-to-point communication between two devices. It differs from synchronous protocols like I2C and SPI by not requiring a clock line, which simplifies the hardware setup at the cost of a bit more complexity in message timing.

**Key Characteristics:**
- **Wiring**: UART only requires two main communication lines:
  - **TX** (Transmit) – Sends data from one device.
  - **RX** (Receive) – Receives data on the other device.
- **Message Structure**: UART communication starts with a start bit, followed by a specified number of data bits (typically 8), and ends with a stop bit. This setup provides a clear beginning and end for each message.
- **Connection Limitation**: UART is generally designed for direct, two-device connections. While ideal for simple applications, it lacks the network capability of protocols like I2C and SPI.

![The basic UART circuit schematic](5.5.png)

---

**Timing and Synchronization**  
UART communication is **asynchronous**, meaning there’s no clock signal to synchronize the devices. Instead, both devices rely on the timing of the messages, defined by a **baud rate** (bits per second). This rate must be set and agreed upon by both devices for messages to be transmitted correctly.

- **Baud Rate**: Baud rates like 9600 and 115200 are commonly used with Arduino projects and can be set in the `setup()` function. 

**Arduino Serial Class**  
In Arduino, the `Serial` class provides built-in UART functionality, allowing for easy data transmission and reception. The class includes various methods:
- **`Serial.begin(baud_rate)`** – Initializes UART communication at a specified baud rate.
- **`Serial.print()`, `Serial.write()`** – Send data over UART.
- **`Serial.read()`, `Serial.println()`** – Receive data and print messages in the serial monitor.

For example:
```cpp
Serial.begin(9600);  // Sets baud rate to 9600
Serial.print("Hello UART!");  // Sends a message



## I2C Communication with an Inertial Measurement Unit (IMU)

### Communication Protocols in Robotics
Communication protocols enable robots to operate efficiently and exchange complex data across various sensors and processors. In this section, we’ll set up an I2C communication protocol to connect an Inertial Measurement Unit (IMU) to an Arduino. This will help our rover track orientation and navigate around obstacles.

### Understanding the IMU
An IMU is a versatile sensor that detects orientation and movement through:
- **Accelerometers** for measuring acceleration,
- **Gyroscopes** for rotation,
- **Magnetometers** (optional) for magnetic field detection.

By combining data from each sensor, IMUs can estimate movement and orientation. Here, we’ll use our IMU to approximate directional changes using the magnetometer.

### Preparing the IMU for I2C
1. **Attach Header Pins**: Begin by soldering six header pins to the IMU, but only to the side with the VIN, 1V8, GND, SCL, SDA, and INT pins.

    ![Figure 5-6: The IMU with 6 header pins soldered onto one side only](5.6.png)

2. **Positioning on the Breadboard**: Insert the IMU into the mini breadboard, leaving space behind it for jumper wires.

3. **Connecting Power and Communication Wires**:
   - Plug two jumper wires from the IMU’s **VIN** and **GND** pins to the 5V and GND buses on the breadboard.
   
   ![Figure 5-7: IMU placement on the mini breadboard](5.7.png)
   
   - Connect **SCL** and **SDA** pins from the IMU to pins I22 and I23 on the Arduino’s main breadboard. This completes the I2C circuit.
   
   ![Figure 5-8: SDA and SCL jumper wires positioned for Arduino](5.8.png)

### Setting Up the I2C Libraries for the IMU
To communicate with the IMU, we’ll use a custom library specifically designed for the **[ICM20948](https://learn.adafruit.com/adafruit-tdk-invensense-icm-20948-9-dof-imu/arduino)** model from Adafruit.

1. **Install Required Libraries**:
   - In Arduino IDE, download the **Adafruit ICM20X**, **Adafruit BusIO**, and **Adafruit Unified Sensor** libraries.

2. **Add Libraries to Code**:

   ```cpp
   // Include communication and sensor libraries
   #include <Adafruit_ICM20X.h>
   #include <Adafruit_ICM20948.h>
   #include <Adafruit_Sensor.h>
   #include <Wire.h>  // For I2C communication (SDA - A4, SCL - A5)
   
   // Create an IMU object from the ICM20948 class
   Adafruit_ICM20948 icm;
   
   // create icm object from ICM20948 class 
   Adafruit_ICM20948 icm;

   ```

## Initializing the IMU in Code

Now on to the setup() function. We’ll give ourselves some troubleshooting tools in this code  with some print statements so we can track the startup process of our IMU in the serial monitor.  We’ll need to begin UART serial communication with our Arduino to allow this, and that means  setting a baud rate.

```cpp
//--------------------set up ICM20948 IMU--------------------  
Serial.begin(115200); 
while (!Serial) { 
delay(10); 
} 
```

The while loop isn’t strictly necessary, but it gives our serial bus time to start up before we do  anything else. Also notice we set the serial baud rate to 115200. When we open the serial  monitor to view data, we need to specify the baud rate in a drop-down menu on the right side of  the monitor, otherwise everything we see will be gibberish. 
Next, we’ll try to start our I2C communication with the IMU chip. All we need to do is call  icm.begin_I2C to attempt to initialize our device. It isn’t strictly necessary, but we’ll also add  some simple error handling logic that lets us know if initialization is unsuccessful and enters an  infinite loop so the program doesn’t progress any further.

```cpp
Serial.println("Adafruit ICM20948 test"); 
// try to initialize 
if (!icm.begin_I2C()) { 
    Serial.println("failed to find ICM20948 chip"); 
    while (1) { // enter infinite loop if chip is not found 
        delay(10); 
        } 
    } 
Serial.println("ICM20948 found"); 
```

## Using IMU Data for Navigation

With the IMU initialized, let’s create a function to get the robot’s angle from the magnetometer.

1. **Define get_angle_mag()**: This function calculates the robot's orientation based on magnetometer readings.
    ```cpp
    float get_angle_mag() {
        sensors_event_t accel, gyro, mag, temp;
        icm.getEvent(&accel, &gyro, &temp, &mag);
        float angle = atan2(-mag.magnetic.y, mag.magnetic.x) * RAD_TO_DEG;
        return angle;
    }
    ```
2. **Calculate Pitch Using Accelerometer**: Try completing `get_()` to get the rover’s pitch by measuring gravity with the accelerometer.

![Example of how the magnetometer would describe a certain magnetic field](5.9.png)


In the above figure, the [ICM20948 datasheet](https://invensense.tdk.com/wp-content/uploads/2024/03/DS-000189-ICM-20948-v1.6.pdf) shows us the axes of the magnetometer compared to the  housing of the chip. When looking from above such that the dot on the ICM20948 chip is in the  top left corner, the x-axis points to the right, the y-axis points straight down, and the z-axis points  [away from you (into the page)](https://physics.stackexchange.com/questions/302386/confusion-regarding-plane-of-the-paper) following the [right-hand-rule](https://en.wikipedia.org/wiki/Right-hand_rule).  

What the magnetic field sensor does is measure magnetic field strength in [micro Teslas](https://en.wikipedia.org/wiki/Tesla_(unit)) along the  x, y, and z axes of the magnetometer. This gives you the [vector components](https://www.varsitytutors.com/hotmath/hotmath_help/topics/components-of-a-vector) of the local  magnetic field. 

The vector components of the magnetic field are useful to us because we can use the [trigonometric function arctan(y/x)](https://en.wikipedia.org/wiki/Inverse_trigonometric_functions) to find the angle of the magnetic field vector. If we can  determine the direction of the magnetic field of the earth relative to our robot, we can compare  the orientation of our robot to something that doesn’t move ([quickly, at least](https://en.wikipedia.org/wiki/Magnetic_declination)).  
So, lets plug y = -4 and x = -3 into arctan and see what happens: 

![Arctan() alone is not sufficient for finding the angle of a vector](5.10.png)

That’s strange, when we use arctan to find the direction of the magnetic field vector, it gives us a  value that's completely opposite that 5uT local magnetic field we measured above (remember the  direction of the vector is measured from the x-axis and in the direction of RRH). How can we  explain this? 

For one thing, when we have two negative vector components, the negatives cancel, and it  appears to the arctan function just the same as having two positive vector components. A similar  thing happens when you have only one negative vector component. In that case, either vector  component could be negative, but arctan will not discriminate between the different possibilities. Essentially, arctan can’t tell the difference between [graph quadrants](https://www.cuemath.com/geometry/quadrant/) 1 and 4 and 2 and 3, so it  assumes your vector is in quadrants 1 and 4. What we need is a version of arctan that can discriminate between these various cases. 

Luckily, Arduino provides the [atan2()](https://en.wikipedia.org/wiki/Atan2) function for this exact situation. This function calculates  the output of arctan, and then considers the signs of the vector components to decide which  quadrant the vector is actually in. We’ll use atan2() for our get_angle() function. A few final  details are atan2() outputs an angle in radians, so we’ll use a conversion factor Arduino provides  called RAD_TO_DEG to get angle in degrees. We’ll also input the negative of the y component  to atan2(), which essentially rotates the axes of our magnetometer so the y-axis points towards  the front of the rover. 

# Your Turn

Complete get_angle_mag() and test it out. How accurate do the values you’re getting seem?  What might cause the behavior of these values? Are there any ways we could improve the  performance of this sensor? 

```
get_angle_mag(){
    #your code here
    return 0
};
```

We can use similar code to find the robot's pitch by using the accelerometer to detect gravity.  Give it a shot yourself by filling in the get_pitch() function! Heres a hint: accel.acceleration.x  will return the acceleration measured by the accelerometer in its x-axis. 


```
get_pitch(){
    #your code here
    return 0
};
```

## Building an FSM for Obstacle Navigation

Now, let's use the IMU and other sensors to build a Finite State Machine (FSM) for obstacle navigation.

### FSM States and Transitions
1. **States**:
   - `drive_forward`
   - `turn_right`
   - `turn_left`

2. **FSM Structure**: The rover will:
   - Drive forward until an obstacle is detected
   - Turn right, check for clearance, and continue forward if clear; if not, repeat the turn

3. **Example Switch-Case Structure**:
    ```cpp
    switch (current_state) {
        case drive_forward:
            // Check for events
            // Perform actions
            break;
        case turn_right:
            // Check for events
            // Perform actions
            break;
        case turn_left:
            // Check for events
            // Perform actions
            break;
    }
    last_state = current_state;
    current_state = next_state;
    ```
![A state-transition diagram that can achieve our rover’s goal](5.12.png)


## Conclusion

In this lesson, you:

1. **Explored Communication Protocols**:
   - Learned the basics of **communication protocols** for connecting sensors and devices.
   - Covered the **I2C**, **SPI**, and **UART** protocols, understanding their uses, advantages, and limitations.

2. **Integrated an Inertial Measurement Unit (IMU)**:
   - Connected the **ICM20948 IMU** sensor using I2C.
   - Implemented functions to read **magnetometer** and **accelerometer** data, translating it into meaningful directional information.

3. **Enhanced Rover Navigation with FSMs**:
   - Created a more sophisticated **Finite State Machine (FSM)** to handle obstacle avoidance.
   - Used sensor input to guide the rover’s navigation around obstacles.

This lesson introduced advanced sensing capabilities and enhanced control structures, setting a strong foundation for future projects that require complex sensor integration and autonomous navigation. Excellent work!
